## はじめに

このノートブックでは、GlacierStyle ECサイトのデータをSnowflakeにインポートします。

**前提条件:**
- setup.sql が実行済みであること（データベース・スキーマ・ステージ・GitHub連携の設定完了）

**主な処理内容:**
- テーブル定義の作成（11テーブル）
- CSV/JSONデータのインポート
- 音声ファイルの文字起こし（AI_TRANSCRIBE）
- PDFドキュメントの解析（AI_PARSE_DOCUMENT）

**生成されるテーブル:**

*ディメンションテーブル（マスタデータ）*
- `dim_customers`: 顧客マスタ（100件）
- `dim_products`: 商品マスタ（576件）

*ファクトテーブル（トランザクションデータ）*
- `fact_orders`: EC取引データ（500件）
- `fact_payments`: クレジット決済情報（360件）
- `fact_web_logs`: Webアクセスログ（14,532件）

*非構造化データテーブル*
- `raw_sns_mentions`: SNS投稿データ（300件）
- `raw_voice_logs`: 音声ログメタデータ（10件）
- `raw_ad_creatives`: 広告クリエイティブ（15件）
- `raw_faq_documents`: FAQドキュメント（PDF解析済み）
- `raw_operation_manuals`: 運営マニュアル（PDF解析済み）

In [1]:
%%sql -r result_env_setup
-- ============================================================================
-- 環境設定
-- ============================================================================
-- 使用するウェアハウスとスキーマを設定
USE WAREHOUSE GLACIERSTYLE_WH;
USE SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

images_dir = 'images/part1/'

def display_image(image_file: str) -> None:
    image_path = os.path.join(images_dir, image_file)
    img = Image.open(image_path)
    plt.figure(figsize=(15, 12))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

In [ ]:
display_image('architecture.png')

## 1. テーブル定義の作成

データをインポートするためのテーブルを作成します。

**テーブル構成:**
- ディメンションテーブル（マスタデータ）: 2テーブル
- ファクトテーブル（トランザクションデータ）: 3テーブル
- 非構造化データテーブル: 5テーブル

In [ ]:
%%sql -r dataframe_1
-- テーブル定義セクションのテーブル
DROP TABLE IF EXISTS dim_customers;
DROP TABLE IF EXISTS dim_products;
DROP TABLE IF EXISTS fact_orders;
DROP TABLE IF EXISTS fact_payments;
DROP TABLE IF EXISTS fact_web_logs;
DROP TABLE IF EXISTS raw_sns_mentions;
DROP TABLE IF EXISTS raw_voice_logs;
DROP TABLE IF EXISTS raw_ad_creatives;

-- データインポートセクションで作成されるテーブル
DROP TABLE IF EXISTS raw_faq_documents;
DROP TABLE IF EXISTS raw_operation_manuals;

In [23]:
%%sql -r result_dim_customers
-- ============================================================================
-- ディメンションテーブル（マスタデータ）
-- ============================================================================

-- 顧客マスタテーブル（dim_customers）
CREATE OR REPLACE TABLE dim_customers (
    customer_id VARCHAR PRIMARY KEY,
    email VARCHAR,
    phone VARCHAR,
    last_name VARCHAR,
    first_name VARCHAR,
    gender VARCHAR,
    birth_date DATE,
    postal_code VARCHAR,
    prefecture VARCHAR,
    city VARCHAR,
    address VARCHAR,
    registration_date DATE,
    membership_tier VARCHAR,
    total_orders INTEGER,
    total_spent DECIMAL(12,2),
    last_order_date DATE,
    email_opt_in BOOLEAN,
    app_installed BOOLEAN
);

-- 商品マスタテーブル（dim_products）
CREATE OR REPLACE TABLE dim_products (
    product_id VARCHAR PRIMARY KEY,
    product_name VARCHAR,
    product_name_en VARCHAR,
    category_l1 VARCHAR,
    category_l2 VARCHAR,
    category_l3 VARCHAR,
    brand VARCHAR,
    supplier_id VARCHAR,
    cost_price DECIMAL(10,2),
    list_price DECIMAL(10,2),
    current_price DECIMAL(10,2),
    stock_quantity INTEGER,
    product_status VARCHAR,
    launch_date DATE,
    description TEXT,
    weight_g INTEGER,
    dimensions VARCHAR
);
-- ============================================================================
-- ファクトテーブル（トランザクションデータ）
-- ============================================================================
-- EC取引データテーブル（fact_orders）
CREATE OR REPLACE TABLE fact_orders (
    order_id VARCHAR PRIMARY KEY,
    order_datetime TIMESTAMP,
    customer_id VARCHAR,
    product_id VARCHAR,
    quantity INTEGER,
    unit_price DECIMAL(10,2),
    discount_amount DECIMAL(10,2),
    tax_amount DECIMAL(10,2),
    total_amount DECIMAL(10,2),
    payment_method VARCHAR,
    shipping_address_id VARCHAR,
    order_channel VARCHAR,
    campaign_id VARCHAR,
    order_status VARCHAR
);

-- クレジット決済情報テーブル（fact_payments）
CREATE OR REPLACE TABLE fact_payments (
    payment_id VARCHAR PRIMARY KEY,
    order_id VARCHAR,
    payment_datetime TIMESTAMP,
    card_brand VARCHAR,
    card_last4 VARCHAR,
    payment_amount DECIMAL(10,2),
    authorization_code VARCHAR,
    payment_status VARCHAR,
    fraud_score DECIMAL(5,2),
    device_fingerprint VARCHAR,
    ip_address VARCHAR,
    billing_country VARCHAR
);

-- Webアクセスログテーブル（fact_web_logs）
CREATE OR REPLACE TABLE fact_web_logs (
    log_id VARCHAR PRIMARY KEY,
    session_id VARCHAR,
    customer_id VARCHAR,
    event_timestamp TIMESTAMP,
    event_type VARCHAR,
    page_url VARCHAR,
    page_category VARCHAR,
    referrer_url VARCHAR,
    utm_source VARCHAR,
    utm_medium VARCHAR,
    utm_campaign VARCHAR,
    device_type VARCHAR,
    browser VARCHAR,
    os VARCHAR,
    time_on_page INTEGER,
    product_id VARCHAR
);
-- ============================================================================
-- 非構造化データテーブル
-- ============================================================================
-- SNS生ログテーブル（raw_sns_mentions）
CREATE OR REPLACE TABLE raw_sns_mentions (
    post_id VARCHAR PRIMARY KEY,
    platform VARCHAR,
    post_type VARCHAR,
    username VARCHAR,
    display_name VARCHAR,
    content VARCHAR,
    posted_at TIMESTAMP,
    likes INTEGER,
    retweets INTEGER,
    replies INTEGER,
    hashtags ARRAY,
    mentioned_products ARRAY,
    media_urls ARRAY
);

-- カスタマー音声ログテーブル（raw_voice_logs）
CREATE OR REPLACE TABLE raw_voice_logs (
    call_id VARCHAR PRIMARY KEY,
    scenario_id VARCHAR,
    audio_file VARCHAR,
    call_duration_sec NUMBER(10,2),
    call_start_time TIMESTAMP,
    call_end_time TIMESTAMP,
    category VARCHAR,
    agent_id VARCHAR,
    customer_phone VARCHAR,
    customer_id VARCHAR,
    call_type VARCHAR,
    transcribed_text TEXT
);

-- 広告クリエイティブテーブル（raw_ad_creatives）
CREATE OR REPLACE TABLE raw_ad_creatives (
    creative_id VARCHAR PRIMARY KEY,
    campaign_id VARCHAR,
    creative_name VARCHAR,
    creative_type VARCHAR,
    image_file_path VARCHAR,
    copy_text TEXT,
    headline VARCHAR,
    cta_text VARCHAR,
    target_segment VARCHAR,
    platform VARCHAR,
    start_date DATE,
    end_date DATE,
    impressions INTEGER,
    clicks INTEGER,
    conversions INTEGER,
    spend DECIMAL(10,2)
);

## 2. データのインポート

ステージに格納されたファイルからデータをインポートします。

**データソース:**
- CSV: 顧客・商品・注文・決済・広告
- JSON: Webログ・SNSログ・音声メタデータ
- PDF: FAQドキュメント・運営マニュアル
- MP3: 音声ファイル（AI_TRANSCRIBEで文字起こし）

### 2-1. CSVデータのインポート

In [31]:
%%sql -r result_import_customers
-- ============================================================================
-- CSVデータのインポート
-- ============================================================================

-- 顧客マスタのインポート（100件）
COPY INTO dim_customers 
  FROM @DATA_STAGE/customers.csv 
  FILE_FORMAT = (TYPE = 'CSV' SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"');

-- 商品マスタのインポート（576件）
COPY INTO dim_products 
  FROM @DATA_STAGE/products.csv 
  FILE_FORMAT = (TYPE = 'CSV' SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"');

-- EC取引データのインポート（500件）
COPY INTO fact_orders 
  FROM @DATA_STAGE/orders.csv 
  FILE_FORMAT = (TYPE = 'CSV' SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"');

-- クレジット決済情報のインポート（360件）
COPY INTO fact_payments 
  FROM @DATA_STAGE/payments.csv 
  FILE_FORMAT = (TYPE = 'CSV' SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"');

-- 広告クリエイティブのインポート（15件）
COPY INTO raw_ad_creatives 
  FROM @DATA_STAGE/ad_creatives.csv 
  FILE_FORMAT = (TYPE = 'CSV' SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"');

### 2-2. JSONデータのインポート

In [36]:
%%sql -r result_import_weblogs
-- ============================================================================
-- JSONデータのインポート
-- ============================================================================

-- JSONフォーマットの定義
CREATE OR REPLACE FILE FORMAT json_format
  TYPE = 'JSON'
  STRIP_OUTER_ARRAY = TRUE;

-- Webアクセスログのインポート（14,532件）
INSERT INTO fact_web_logs (
    log_id, 
    session_id, 
    customer_id, 
    event_timestamp, 
    event_type, 
    page_url,
    page_category,
    referrer_url,
    utm_source,
    utm_medium,
    utm_campaign,
    device_type,
    browser,
    os,
    time_on_page,
    product_id
)
SELECT 
    $1:log_id::VARCHAR,
    $1:session_id::VARCHAR,
    $1:customer_id::VARCHAR,
    $1:event_timestamp::TIMESTAMP,
    $1:event_type::VARCHAR,
    $1:page_url::VARCHAR,
    $1:page_category::VARCHAR,
    $1:referrer_url::VARCHAR,
    $1:utm_source::VARCHAR,
    $1:utm_medium::VARCHAR,
    $1:utm_campaign::VARCHAR,
    $1:device_type::VARCHAR,
    $1:browser::VARCHAR,
    $1:os::VARCHAR,
    $1:time_on_page::VARCHAR,
    $1:product_id::VARCHAR
FROM @DATA_STAGE/web_logs.json
(FILE_FORMAT => json_format);

-- SNS生ログのインポート（300件）
INSERT INTO raw_sns_mentions (
    post_id,
    platform,
    post_type,
    username,
    display_name,
    content,
    posted_at,
    likes,
    retweets,
    replies,
    hashtags,
    mentioned_products,
    media_urls
)
SELECT 
    $1:post_id::VARCHAR,
    $1:platform::VARCHAR,
    $1:post_type::VARCHAR,
    $1:username::VARCHAR,
    $1:display_name::VARCHAR,
    $1:content::VARCHAR,
    $1:posted_at::TIMESTAMP,
    $1:likes::INTEGER,
    $1:retweets::INTEGER,
    $1:replies::INTEGER,
    $1:hashtags::ARRAY,
    $1:mentioned_products::ARRAY,
    $1:media_urls::ARRAY
FROM @DATA_STAGE/sns_logs.json
(FILE_FORMAT => json_format);

### 2-3. 音声データのインポートと文字起こし

コールセンターの音声ファイル（MP3）をAI_TRANSCRIBEで文字起こしします。

In [ ]:
display_image('ai_transcribe.png')

In [38]:
%%sql -r result_import_voice_meta
-- ============================================================================
-- カスタマー音声ログのメタデータインポートと文字起こし
-- ============================================================================

-- 音声ログメタデータのインポート（10件）
INSERT INTO raw_voice_logs (
    call_id, 
    scenario_id,
    audio_file,
    call_duration_sec,
    call_start_time, 
    call_end_time,
    category,
    agent_id,
    customer_phone,
    customer_id,
    call_type
)
SELECT
    $1:call_id::VARCHAR, 
    $1:scenario_id::VARCHAR,
    $1:audio_file::VARCHAR,
    $1:call_duration_sec::NUMBER(10,2),
    $1:call_start_time::TIMESTAMP, 
    $1:call_end_time::TIMESTAMP,
    $1:category::VARCHAR,
    $1:agent_id::VARCHAR,
    $1:customer_phone::VARCHAR,
    $1:customer_id::VARCHAR,
    $1:call_type::VARCHAR
FROM @DATA_STAGE/voice_logs/voice_logs_metadata.json
(FILE_FORMAT => json_format);

-- ステージのリフレッシュ
ALTER STAGE DATA_STAGE REFRESH;

-- MP3ファイルをAI_TRANSCRIBEで処理し、transcribed_textカラムを更新
MERGE INTO raw_voice_logs AS target
USING (
    SELECT 
        SPLIT_PART(relative_path, '/', -1) AS file_name,
        AI_TRANSCRIBE(
            TO_FILE('@DATA_STAGE', relative_path)
        ):text::TEXT AS transcribed_text
    FROM DIRECTORY(@DATA_STAGE)
    WHERE REGEXP_LIKE(relative_path, 'voice_logs.*\.mp3', 'i')
) AS source
ON target.audio_file = source.file_name
WHEN MATCHED THEN
    UPDATE SET target.transcribed_text = source.transcribed_text;

### 2-4. PDFドキュメントの解析

FAQドキュメントと運営マニュアル（PDF）をAI_PARSE_DOCUMENTで解析し、マークダウンヘッダーでチャンク分割します。

In [ ]:
display_image('ai_parse_document.png')

In [ ]:
%%sql -r dataframe_3
-- ============================================================================
-- PDFドキュメントの解析とインポート
-- ============================================================================

-- ステージをリフレッシュして最新のファイルを認識
ALTER STAGE DATA_STAGE REFRESH;

-- FAQドキュメント（PDF）をAI_PARSE_DOCUMENTで解析
CREATE OR REPLACE TABLE raw_faq_documents AS
WITH parsed_doc AS (
    SELECT 
        *, 
        AI_PARSE_DOCUMENT(
            TO_FILE('@DATA_STAGE', relative_path),
            {'mode': 'LAYOUT', 'page_split': false}
        ) AS contents
    FROM DIRECTORY(@DATA_STAGE)
    WHERE LOWER(relative_path) = 'faq_document.pdf'
)
SELECT * FROM parsed_doc;

-- 運営マニュアル（PDF）をAI_PARSE_DOCUMENTで解析
CREATE OR REPLACE TABLE raw_operation_manuals AS
WITH parsed_doc AS (
    SELECT 
        *, 
        AI_PARSE_DOCUMENT(
            TO_FILE('@DATA_STAGE', relative_path),
            {'mode': 'LAYOUT', 'extract_images': true, 'page_split': false}
        ) AS contents
    FROM DIRECTORY(@DATA_STAGE)
    WHERE LOWER(relative_path) = 'operation_manual_w_images.pdf'
) 
SELECT * FROM parsed_doc;

## 3. データ確認

各テーブルのレコード数を確認します。

In [49]:
%%sql -r result_verify
-- ============================================================================
-- 各テーブルのレコード数を確認
-- ============================================================================
SELECT 
    'dim_customers' AS table_name, 
    COUNT(*) AS record_count,
    '顧客マスタ' AS description
FROM dim_customers
UNION ALL
SELECT 
    'dim_products' AS table_name, 
    COUNT(*) AS record_count,
    '商品マスタ' AS description
FROM dim_products
UNION ALL
SELECT 
    'fact_orders' AS table_name, 
    COUNT(*) AS record_count,
    'EC取引データ' AS description
FROM fact_orders
UNION ALL
SELECT 
    'fact_payments' AS table_name, 
    COUNT(*) AS record_count,
    'クレジット決済情報' AS description
FROM fact_payments
UNION ALL
SELECT 
    'fact_web_logs' AS table_name, 
    COUNT(*) AS record_count,
    'Webアクセスログ' AS description
FROM fact_web_logs
UNION ALL
SELECT 
    'raw_sns_mentions' AS table_name, 
    COUNT(*) AS record_count,
    'SNS生ログ' AS description
FROM raw_sns_mentions
UNION ALL
SELECT 
    'raw_voice_logs' AS table_name, 
    COUNT(*) AS record_count,
    'カスタマー音声ログ' AS description
FROM raw_voice_logs
UNION ALL
SELECT 
    'raw_ad_creatives' AS table_name, 
    COUNT(*) AS record_count,
    '広告クリエイティブ' AS description
FROM raw_ad_creatives
UNION ALL
SELECT 
    'raw_faq_documents' AS table_name, 
    COUNT(*) AS record_count,
    'FAQドキュメント（解析済み）' AS description
FROM raw_faq_documents
UNION ALL
SELECT 
    'raw_operation_manuals' AS table_name, 
    COUNT(*) AS record_count,
    '運営マニュアル（解析済み）' AS description
FROM raw_operation_manuals
ORDER BY table_name;

## まとめ

このノートブックでは、GlacierStyle ECサイトのデータをSnowflakeにインポートしました。

### 作成したテーブル一覧

**ディメンションテーブル（マスタデータ）**
- `dim_customers`: 顧客マスタ
- `dim_products`: 商品マスタ

**ファクトテーブル（トランザクションデータ）**
- `fact_orders`: EC取引データ
- `fact_payments`: クレジット決済情報
- `fact_web_logs`: Webアクセスログ

**非構造化データテーブル**
- `raw_sns_mentions`: SNS投稿データ
- `raw_voice_logs`: 音声ログ（文字起こし済み）
- `raw_ad_creatives`: 広告クリエイティブ
- `raw_faq_documents`: FAQドキュメント（PDF解析済み）
- `raw_operation_manuals`: 運営マニュアル（PDF解析済み）

### 使用したCortex AI関数

- `AI_TRANSCRIBE`: 音声ファイルの文字起こし
- `AI_PARSE_DOCUMENT`: PDFドキュメントの解析

### 次のステップ

- **Part 2**: データの加工・変換（AI_EXTRACT, AI_SENTIMENT, AI_CLASSIFY等）
- **Part 3**: メタデータの自動付与とセマンティックビュー作成